# Understanding and Codin gthe Self-Attention of LLM from Scratch

In this article, we are going to understand how self-attention works from scratch. This means we will code it ourselves one step at a time.

Since its introduction via the original transformer paper (Attention Is All You Need), self-attention has become a cornerstone of many state-of-the-art deep learning models, particularly in the field of Natural Language Processing (NLP). Since self-attention is now everywhere, it’s important to understand how it works.

![](image1.png)

## Self-Attention

The concept of “attention” in deep learning has its roots in the effort to improve Recurrent Neural Networks (RNNs) for handling longer sequences or sentences. For instance, consider translating a sentence from one language to another. Translating a sentence word-by-word does not work effectively.

![](image2.png)

To overcome this issue, attention mechanisms were introduced to give access to all sequence elements at each time step. The key is to be selective and determine which words are most important in a specific context. In 2017, the transformer architecture introduced a standalone self-attention mechanism, eliminating the need for RNNs altogether.

![](image3.png)

We can think of self-attention as a mechanism that enhances the information content of an input embedding by including information about the input’s context. In other words, the self-attention mechanism enables the model to weigh the importance of different elements in an input sequence and dynamically adjust their influence on the output. This is especially important for language processing tasks, where the meaning of a word can change based on its context within a sentence or document.

Note that there are many variants of self-attention. A particular focus has been on making self-attention more efficient. However, most papers still implement the original scaled-dot product attention mechanism discussed in this paper since it usually results in superior accuracy and because self-attention is rarely a computational bottleneck for most companies training large-scale transformers

n this article, we focus on the original scaled-dot product attention mechanism (referred to as self-attention), which remains the most popular and most widely used attention mechanism in practice. However, if you are interested in other types of attention mechanisms, check out the 2020 Efficient Transformers: A Survey and the 2023 A Survey on Efficient Training of Transformers review and the recent FlashAttention paper.

## Embedding an Input Sentence

Before we begin, let’s consider an input sentence “Life is short, eat dessert first” that we want to put through the self-attention mechanism. Similar to other types of modeling approaches for processing text (e.g., using recurrent neural networks or convolutional neural networks), we create a sentence embedding first.

For simplicity, here our dictionary dc is restricted to the words that occur in the input sentence. In a real-world application, we would consider all words in the training dataset (typical vocabulary sizes range between 30k to 50k).

In [1]:
sentence = 'Life is short, eat dessert first'

dc = {s:i for i,s in enumerate(sorted(sentence.replace(',', '').split()))}
print(dc)

{'Life': 0, 'dessert': 1, 'eat': 2, 'first': 3, 'is': 4, 'short': 5}


Next, we use this dictionary to assign an integer index to each word:

In [2]:
import torch

sentence_int = torch.tensor(
    [
        dc[s] for s in sentence.replace(',','').split()
    ]
)

In [3]:
print(sentence_int)

tensor([0, 4, 5, 2, 1, 3])


Now, using the integer-vector representation of the input sentence, we can use an embedding layer to encode the inputs into a real-vector embedding. Here, we will use a 16-dimensional embedding such that each input word is represented by a 16-dimensional vector. Since the sentence consists of 6 words, this will result in a 6×16-dimensional embedding:

In [4]:
torch.manual_seed(123)
embed = torch.nn.Embedding(6, 16)
embedded_sentence = embed(sentence_int).detach()

In [5]:
print(embedded_sentence)

tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  0.3486,  0.6603, -0.2196, -0.3792,
          0.7671, -1.1925,  0.6984, -1.4097,  0.1794,  1.8951,  0.4954,  0.2692],
        [ 0.5146,  0.9938, -0.2587, -1.0826, -0.0444,  1.6236, -2.3229,  1.0878,
          0.6716,  0.6933, -0.9487, -0.0765, -0.1526,  0.1167,  0.4403, -1.4465],
        [ 0.2553, -0.5496,  1.0042,  0.8272, -0.3948,  0.4892, -0.2168, -1.7472,
         -1.6025, -1.0764,  0.9031, -0.7218, -0.5951, -0.7112,  0.6230, -1.3729],
        [-1.3250,  0.1784, -2.1338,  1.0524, -0.3885, -0.9343, -0.4991, -1.0867,
          0.8805,  1.5542,  0.6266, -0.1755,  0.0983, -0.0935,  0.2662, -0.5850],
        [-0.0770, -1.0205, -0.1690,  0.9178,  1.5810,  1.3010,  1.2753, -0.2010,
          0.4965, -1.5723,  0.9666, -1.1481, -1.1589,  0.3255, -0.6315, -2.8400],
        [ 0.8768,  1.6221, -1.4779,  1.1331, -1.2203,  1.3139,  1.0533,  0.1388,
          2.2473, -0.8036, -0.2808,  0.7697, -0.6596, -0.7979,  0.1838,  0.2293]])


In [6]:
print(embedded_sentence.shape)

torch.Size([6, 16])


## Defining the Weight Matrcies

Now, let’s discuss the widely utilized self-attention mechanism known as the scaled dot-product attention, which is integrated into the transformer architecture.

Self-attention utilizes three weight matrices, referred to as $W_q$, $W_k$, and $W_v$, which are adjusted as model parameters during training. These matrices serve to project the inputs into query, key, and value components of the sequence, respectively.

The respective query, key and value sequences are obtained via matrix multiplication between the weight matrices $W$ and the embedded inputs $x$:
- Query Sequence: $q^{(i)} = W_q.x^{(i)}$ for $i \in [1, T]$
- Key Sequence: $k^{(i)} = W_k.x^{(i)}$ for $i \in [1, T]$
- Value Sequence: $v^{(i)} = W_v.x^{(i)}$ for $i \in [1, T]$

The index $i$ refers to the token index position in the input sequence, which has length $T$.

![](image4.png)

Here both $q^{(i)}$ and $k^{(i)}$ are vectors of dimension $d_k$. The projection matrices $W_q$ and $W_k$ have a shape of $d_k×d$, while $W_v$ has the shape $d_v×d$.

(It’s important to note that $d$ represents the size of each word vector, $x$.)

Since we are computing the dot-product between the query and key vectors, these two vectors have to contain the same number of elements ($d_q=d_k$). However, the number of elements in the value vector $v^{(i)}$, which determines the size of the resulting context vector, is arbitrary.

So, for the following code walkthrough, we will set $d_q=d_k=24$ and use $d_v=28$, initializing the projection matrices as follows:

In [7]:
torch.manual_seed(123)

d = embedded_sentence.shape[1]

d

16

In [8]:
d_q, d_k, d_v = 24, 24, 28

In [10]:
W_query = torch.nn.Parameter(torch.rand(d_q, d))

W_query.shape

torch.Size([24, 16])

In [11]:
W_query

Parameter containing:
tensor([[0.0204, 0.8290, 0.1063, 0.2062, 0.5058, 0.6522, 0.7905, 0.4298, 0.2427,
         0.4570, 0.6638, 0.2187, 0.0657, 0.7387, 0.1691, 0.2186],
        [0.9148, 0.1705, 0.0943, 0.8800, 0.2614, 0.5325, 0.9981, 0.3005, 0.9657,
         0.8973, 0.8862, 0.6483, 0.2746, 0.8148, 0.1575, 0.2087],
        [0.2590, 0.7162, 0.5689, 0.8181, 0.8286, 0.5292, 0.7914, 0.1387, 0.0221,
         0.0927, 0.7759, 0.9598, 0.3617, 0.7766, 0.1427, 0.4906],
        [0.4970, 0.3552, 0.2576, 0.7346, 0.4564, 0.4009, 0.8474, 0.1203, 0.8265,
         0.9441, 0.1928, 0.0263, 0.5696, 0.1197, 0.7091, 0.1012],
        [0.1098, 0.6353, 0.3719, 0.0574, 0.6951, 0.6766, 0.5674, 0.8267, 0.2993,
         0.9564, 0.1189, 0.9508, 0.8715, 0.0552, 0.4556, 0.2310],
        [0.9920, 0.4791, 0.7945, 0.9323, 0.1144, 0.8039, 0.0651, 0.3650, 0.2984,
         0.0324, 0.0290, 0.0179, 0.1132, 0.2206, 0.3352, 0.7797],
        [0.4196, 0.0050, 0.1368, 0.8588, 0.0121, 0.2541, 0.0475, 0.7690, 0.8418,
         0.5438

In [12]:
W_key = torch.nn.Parameter(torch.rand(d_k, d))
W_value = torch.nn.Parameter(torch.rand(d_v, d))

## Computing the Unnormalized Attention Weights

Now, let’s suppose we are interested in computing the attention-vector for the second input element – the second input element acts as the query here:

![](image5.png)

In [13]:
x_2 = embedded_sentence[1]
x_2

tensor([ 0.5146,  0.9938, -0.2587, -1.0826, -0.0444,  1.6236, -2.3229,  1.0878,
         0.6716,  0.6933, -0.9487, -0.0765, -0.1526,  0.1167,  0.4403, -1.4465])

In [14]:
query_2 = W_query.matmul(x_2)

query_2

tensor([-0.0809, -1.2746, -2.3948, -0.3425,  1.5967,  0.5399,  0.9113,  0.0962,
         0.7300, -1.0553,  1.2533, -0.2113,  1.0208, -0.7470,  1.5171,  0.2773,
        -0.3173,  0.2698,  1.5237, -1.0970,  1.3849,  0.4400, -2.4926,  0.3594],
       grad_fn=<MvBackward0>)

In [15]:
key_2 = W_key.matmul(x_2)
value_2 = W_value.matmul(x_2)

In [16]:
print(query_2.shape)
print(key_2.shape)
print(value_2.shape)

torch.Size([24])
torch.Size([24])
torch.Size([28])


We can then generalize this to compute th remaining key, and value elements for all inputs as well, since we will need them in the next step when we compute the unnormalized attention weights $ω$:

In [17]:
keys = W_key.matmul(embedded_sentence.T).T
values = W_value.matmul(embedded_sentence.T).T

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 24])
values.shape: torch.Size([6, 28])


Now that we have all the required keys and values, we can proceed to the next step and compute the unnormalized attention weights $ω$ , which are illustrated in the figure below:

![](image6.png)

As illustrated in the figure above, we compute $ω_{i,j}$ as the dot product between the query and key sequences:

$$ω_{i,j}=q^{(i)^T}.k^{(j)}$$

For example, we can compute the unnormalized attention weight for the query and 5th input element (corresponding to index position 4) as follows:

In [20]:
query_2

tensor([-0.0809, -1.2746, -2.3948, -0.3425,  1.5967,  0.5399,  0.9113,  0.0962,
         0.7300, -1.0553,  1.2533, -0.2113,  1.0208, -0.7470,  1.5171,  0.2773,
        -0.3173,  0.2698,  1.5237, -1.0970,  1.3849,  0.4400, -2.4926,  0.3594],
       grad_fn=<MvBackward0>)

In [21]:
keys[4]

tensor([-3.1399, -0.6158,  1.3958, -0.7103, -0.8951, -1.7725, -0.2955, -2.7453,
         0.7757,  1.4735,  0.0459, -1.9004, -0.6969,  1.2049, -0.9530, -4.2561,
         0.8399, -3.7423, -1.1625, -2.9441,  0.7369, -1.5681, -1.7600, -0.3928],
       grad_fn=<SelectBackward0>)

In [19]:
omega_24 = query_2.dot(keys[4])

omega_24

tensor(-4.9889, grad_fn=<DotBackward0>)

Since we will need those to compute the attention scores later, let’s compute the $ω$ values for all input tokens as illustrated in the previous figure:

In [22]:
omega_2 = query_2.matmul(keys.T)

omega_2

tensor([ 1.5749, -2.1533, -5.0682, -0.4960, -4.9889, -0.3455],
       grad_fn=<SqueezeBackward4>)

## Computing the Attention Scores

The subsequent step in self-attention is to normalize the unnormalized attention weights, $ω$, to obtain the normalized attention weights, $α$, by applying the $softmax$ function. Additionally, $\frac{1}{\sqrt{d_k}}$ is used to scale $ω$ before normalizing it through the softmax function, as shown below:

![](image7.png)

The scaling by the square-root of $d_k$ ensures that the $Euclidean$ length of the weight vectors will be approximately in the same magnitude. This helps prevent the attention weights from becoming too small or too large, which could lead to numerical instability or affect the model’s ability to converge during training.

Why does specifically $\sqrt{d_k}$? The dot product between q and k is a sum of dk independent terms, each with variance about 1. That means the variance of the raw score grows linearly with $d_k$. By dividing by $\sqrt{d_k}$, we cancel that growth and bring the variance back to about 1.

In code, we can implement the computation of the attention weights as follows:



In [23]:
import torch.nn.functional as F

attention_weights_2 = F.softmax(omega_2 / d_k**0.5, dim=0)

attention_weights_2

tensor([0.3014, 0.1408, 0.0777, 0.1975, 0.0789, 0.2037],
       grad_fn=<SoftmaxBackward0>)

Finally, the last step is to compute the context vector $z^(2)$, which is an attention-weighted version of our original query input $x^(2)$, including all the other input elements as its context via the attention weights:

![](image8.png)

In [24]:
context_vector_2 = attention_weights_2.matmul(values)

print(context_vector_2.shape)
print(context_vector_2)

torch.Size([28])
tensor([-0.0758,  0.9186,  0.5830,  1.4539,  0.6618,  0.1828,  0.2682, -0.0466,
         0.1750,  0.1124, -0.2370,  0.6784,  0.6392, -0.1123, -0.3668,  0.8103,
         0.0537,  0.2456,  0.1256,  0.5627, -0.3052, -0.0980, -0.2289, -0.3689,
        -0.1497, -0.1664,  0.7972,  0.3714], grad_fn=<SqueezeBackward4>)


Note that this output vector has more dimensions $(d_v=28)$ than the original input vector $(d=16)$ since we specified $d_v>d$ earlier; however, the embedding size choice is arbitrary.